# Challenge de la Sesión 03 — Modelos preentrenados y transferencia de aprendizaje
**Desarrollo y Evaluación de Modelos Propios** · Especialización en IA Generativa y Desarrollo de Negocios · Universidad del Rosario · Rosario GSB

[![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/03-transfer-learning/challenge.ipynb)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/03-transfer-learning/challenge.ipynb)

**Pregunta central: ¿cuántos datos etiquetados necesitas, y cuándo basta con un prompt?** Vas a medir una curva de aprendizaje con embeddings congelados, compararla con un LLM pequeño en modo *few-shot* sobre tu propio corte de noticias y convertir tus números en una recomendación para un caso de negocio.

| | |
|---|---|
| **Modalidad** | Individual. Tu configuración (tamaño de entrenamiento, número de ejemplos en el prompt, corte de prueba y caso de negocio) depende de tu código estudiantil. |
| **Tiempo** | 40 minutos en clase; entrega hasta las **23:59 de hoy**. |
| **Uso de IA** | Permitido, con **registro obligatorio** al final. Sin registro, el componente de reflexión (20%) vale 0.0. |
| **Micro-sustentaciones** | El profesor puede pedirte que expliques tu entrega en 2–3 minutos, sin ayuda de IA. Si no puedes explicarla, la nota máxima de este challenge es 3.0. |
| **Rúbrica** | Ejecución técnica 40% · Análisis e interpretación 40% · Reflexión y registro de IA 20%. |
| **Entrega** | *File → Download notebook* (Archivo → Descargar notebook) y súbelo a e-Aulas como `S03_<codigo>.ipynb`. |

**Antes de empezar:** en Kaggle activa *Settings → Internet → On*. Si tienes cuota, activa *Accelerator → GPU T4 x2*: la Tarea 2 corre mucho más rápido (en CPU también funciona). Escribe tu código estudiantil en la primera celda y ejecuta todo en orden.

**Recuerda también:** hoy se cierra el hito H1 del proyecto final (registro del grupo de 3 como `PF_H1_<codigo>`, 23:59).

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S03"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # los modelos y datos se descargan de Hugging Face
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "Sin conexión a internet: este notebook descarga modelos y datos de Hugging Face.\n"
            "Kaggle: en el panel derecho abre Settings (Session options) y activa Internet. "
            "La opción solo aparece si tu cuenta tiene el teléfono verificado (kaggle.com/settings). "
            "Luego vuelve a ejecutar esta celda. En Colab el internet viene activado. "
            "En tu propio computador: revisa tu conexión, o define HF_HUB_OFFLINE=1 si ya tienes los modelos descargados."
        ) from None

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---
ensure("transformers")
ensure("datasets")
ensure("scikit-learn", "sklearn")
ensure("pandas")
ensure("matplotlib")

In [ ]:
# Casos de negocio (empresas ficticias). Cada estudiante recibe uno.
BUSINESS_CASES = [
    {"empresa": "Andes Bank (banco digital)",
     "tarea": "Enrutar los correos de clientes a 8 áreas (tarjetas, créditos, fraude, PQRS…)",
     "datos_etiquetados": "120 correos etiquetados (15 por área)",
     "volumen_mensual": 40_000, "latencia": "menos de 2 s por correo",
     "restricciones": "datos personales: no pueden salir de la nube privada del banco; textos en español"},
    {"empresa": "NovaTel (telecomunicaciones)",
     "tarea": "Detectar la intención de mensajes de WhatsApp de soporte (10 intenciones)",
     "datos_etiquetados": "300 mensajes etiquetados (30 por intención)",
     "volumen_mensual": 60_000, "latencia": "menos de 1 s por mensaje",
     "restricciones": "mensajes informales en español colombiano, con emojis y abreviaturas"},
    {"empresa": "Banco Andino",
     "tarea": "Clasificar comentarios de encuestas de satisfacción en positivo, neutral o negativo",
     "datos_etiquetados": "ninguno",
     "volumen_mensual": 500, "latencia": "un lote al mes",
     "restricciones": "no hay equipo técnico; presupuesto mínimo; los comentarios no traen datos personales"},
    {"empresa": "Clínica Serena",
     "tarea": "Asignar solicitudes de citas del portal web a 6 especialidades",
     "datos_etiquetados": "2.000 solicitudes etiquetadas",
     "volumen_mensual": 25_000, "latencia": "menos de 500 ms por solicitud",
     "restricciones": "datos de salud: todo debe correr en los servidores de la clínica"},
    {"empresa": "Editorial Río Claro (medio digital)",
     "tarea": "Etiquetar su archivo histórico de artículos en español en 4 secciones",
     "datos_etiquetados": "64 artículos etiquetados por sección",
     "volumen_mensual": 200_000, "latencia": "un solo lote (se hace una vez)",
     "restricciones": "presupuesto limitado; pueden usar GPU gratuita de Kaggle"},
    {"empresa": "Aseguradora Cóndor",
     "tarea": "Enrutar reclamaciones a 12 tipos de siniestro; las categorías cambian cada trimestre",
     "datos_etiquetados": "30 reclamaciones etiquetadas por tipo",
     "volumen_mensual": 15_000, "latencia": "menos de 5 s por reclamación",
     "restricciones": "el regulador exige poder explicar por qué se enrutó cada caso"},
]

CONFIG = {
    "k_train": pick([16, 64, 256, 1024]),          # ejemplos etiquetados POR CLASE para la regresión logística
    "n_shots": pick([1, 2, 4]),                    # ejemplos resueltos POR CLASE dentro del prompt del LLM
    "classes": ["World", "Sports", "Business", "Sci/Tech"],   # las 4 clases de AG News
    "test_seed": pick(range(1000, 10000)),         # semilla de tu corte personal de prueba
    "business_case": pick(BUSINESS_CASES),
}
print("🎯 Tu configuración personal")
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## Preparación (ejecuta sin cambios)

Esta celda carga AG News, limpia el texto y define las funciones del lab: `embed` (embeddings congelados de `all-MiniLM-L6-v2` con *mean pooling*), `llm_classify` (clasificación con `Qwen/Qwen2.5-0.5B-Instruct` y su plantilla de chat) y `parse_label`. Los tamaños se reducen solo en la prueba automática del curso (`FAST_DEV_RUN`).

In [ ]:
import re, time, gc
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

N_TEST = 40 if FAST_DEV_RUN else 400                     # corte personal para la curva de aprendizaje
N_TEST_LLM = 8 if FAST_DEV_RUN else 40                   # las primeras N del corte, para comparar con el LLM
CURVE_KS = [4, 8, 16] if FAST_DEV_RUN else [16, 64, 256] # puntos de la curva (ejemplos por clase)
K_TRAIN = min(CONFIG["k_train"], 16) if FAST_DEV_RUN else CONFIG["k_train"]
N_SHOTS = 1 if FAST_DEV_RUN else CONFIG["n_shots"]
MAX_NEW_TOKENS = 4 if FAST_DEV_RUN else 6

ag_news = load_dataset("fancyzhx/ag_news")
LABELS = ag_news["train"].features["label"].names
N_CLASSES = len(LABELS)

def _entity(match):
    code = int(match.group(1))
    return chr(code) if code < 128 else " "

def clean(text):
    """Limpia artefactos de AG News: entidades numéricas, 'quot;', barras invertidas y etiquetas HTML."""
    text = re.sub(r"#(\d+);", _entity, text)
    text = text.replace("quot;", '"').replace("\\", " ")
    text = re.sub(r"&lt;.*?&gt;", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def sample_per_class(split, k, seed):
    """k ejemplos al azar de cada clase (ordenados por clase). Misma semilla → muestras anidadas."""
    rng_np = np.random.default_rng(seed)
    labels = np.array(split["label"])
    idx = np.concatenate([rng_np.permutation(np.where(labels == c)[0])[:k] for c in range(N_CLASSES)])
    subset = split.select(idx.tolist())
    return [clean(t) for t in subset["text"]], np.array(subset["label"])

# Tu corte personal de prueba (depende de CONFIG["test_seed"])
test_split = ag_news["test"].shuffle(seed=CONFIG["test_seed"]).select(range(N_TEST))
test_texts, y_test = [clean(t) for t in test_split["text"]], np.array(test_split["label"])

# Codificador congelado para la extracción de características
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"
emb_tok = AutoTokenizer.from_pretrained(EMB_NAME)
encoder = AutoModel.from_pretrained(EMB_NAME).to(DEVICE).eval()

@torch.no_grad()
def embed(texts, batch_size=64):
    """Un vector de 384 números por texto: promedio de los vectores de sus tokens, normalizado."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        enc = emb_tok(texts[i:i + batch_size], padding=True, truncation=True,
                      max_length=128, return_tensors="pt").to(DEVICE)
        hidden = encoder(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        vectors.append(F.normalize(mean, dim=1).cpu())
    return torch.cat(vectors).numpy()

# LLM pequeño para zero-shot y few-shot
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).float().to(DEVICE).eval()  # float32: transformers 5.x carga bfloat16 por defecto
SYSTEM_PROMPT = ("You are a news topic classifier. Read the article and answer with exactly one label "
                 "from this list: World, Sports, Business, Sci/Tech. Answer with the label only.")
LABEL_PATTERNS = {"Sci/Tech": r"sci|tech", "Sports": r"sport", "Business": r"business", "World": r"world"}

def shorten(text, max_words):
    words = text.split()
    return " ".join(words[:max_words]) + (" …" if len(words) > max_words else "")

def build_messages(text, examples=()):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex_text, ex_label in examples:
        messages.append({"role": "user", "content": "Article: " + shorten(ex_text, 40)})
        messages.append({"role": "assistant", "content": LABELS[ex_label]})
    messages.append({"role": "user", "content": "Article: " + shorten(text, 60)})
    return messages

def parse_label(answer):
    """Convierte la respuesta libre en una etiqueta: gana el patrón que aparece primero; si no, 'unknown'."""
    answer = answer.lower()
    best_label, best_pos = "unknown", len(answer) + 1
    for label, pattern in LABEL_PATTERNS.items():
        match = re.search(pattern, answer)
        if match and match.start() < best_pos:
            best_label, best_pos = label, match.start()
    return best_label

@torch.no_grad()
def llm_classify(texts, examples=()):
    """Devuelve etiquetas, respuestas crudas, segundos por texto y longitud del prompt en tokens."""
    labels, answers, seconds, prompt_tokens = [], [], [], []
    for text in texts:
        inputs = llm_tok.apply_chat_template(build_messages(text, examples), add_generation_prompt=True,
                                             return_tensors="pt", return_dict=True).to(DEVICE)
        start = time.perf_counter()
        output = llm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                              pad_token_id=llm_tok.eos_token_id)
        seconds.append(time.perf_counter() - start)
        answer = llm_tok.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        answers.append(answer.strip())
        labels.append(parse_label(answer))
        prompt_tokens.append(inputs["input_ids"].shape[1])
    return labels, answers, seconds, prompt_tokens

print(f"Corte de prueba: {N_TEST} noticias · por clase: {dict(zip(LABELS, np.bincount(y_test, minlength=N_CLASSES)))}")
print(f"k_train = {K_TRAIN} · curva = {CURVE_KS} · n_shots = {N_SHOTS} · prueba LLM = {N_TEST_LLM} noticias · {DEVICE}")

## Tarea 1 — Curva de aprendizaje con embeddings congelados (Ejecución técnica)

Entrena una regresión logística sobre los embeddings congelados con tu `k_train` asignado y con los puntos de la curva (16, 64 y 256 ejemplos por clase). Las muestras están **anidadas**: los 16 ejemplos de una clase están incluidos en los 64, y estos en los 256, así que la curva mide el efecto de *añadir* datos.

1. Ejecuta la celda tal como está (ya funciona con valores razonables). Si tu `k_train` es 1024, calcular los embeddings tarda uno o dos minutos en CPU.
2. ✏️ Completa la gráfica: agrega una línea horizontal con la exactitud de tu clasificador de `k_train` y ponle título y ejes en español.
3. Responde en la celda de texto.

In [ ]:
# Embeddings de entrenamiento: una sola vez para el k más grande (las muestras son anidadas)
K_MAX = max(CURVE_KS + [K_TRAIN])
train_texts_all, y_train_all = sample_per_class(ag_news["train"], K_MAX, seed=PERSONAL_SEED % (2**32))
rank_in_class = np.tile(np.arange(K_MAX), N_CLASSES)      # posición de cada ejemplo dentro de su clase

start = time.perf_counter()
X_train_all = embed(train_texts_all)
X_test = embed(test_texts)
embed_s = time.perf_counter() - start
ms_embed_per_text = 1000 * embed_s / (len(train_texts_all) + len(test_texts))

def train_and_score(k):
    """Regresión logística con k ejemplos por clase: exactitud en tu corte y segundos de entrenamiento."""
    keep = rank_in_class < k
    start = time.perf_counter()
    clf = LogisticRegression(max_iter=1000).fit(X_train_all[keep], y_train_all[keep])
    train_s = time.perf_counter() - start
    return clf, accuracy_score(y_test, clf.predict(X_test)), train_s

curve = {}
for k in sorted(set(CURVE_KS + [K_TRAIN])):
    _, acc_k, train_s = train_and_score(k)
    curve[k] = acc_k
    print(f"k = {k:4d} por clase ({N_CLASSES * k:5d} etiquetas) → exactitud {acc_k:.3f} · entrenamiento {train_s:.2f} s")

clf_k, acc_lr_k, train_s_k = train_and_score(K_TRAIN)
print(f"\nTu clasificador (k_train = {K_TRAIN}): exactitud {acc_lr_k:.3f} en {N_TEST} noticias · "
      f"embeddings: {ms_embed_per_text:.1f} ms por texto")

# ✏️ TU CÓDIGO AQUÍ: completa la gráfica (línea horizontal con acc_lr_k, título y ejes en español)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(list(curve), list(curve.values()), marker="o")
ax.set_xscale("log", base=2)
plt.show()

**Tu respuesta:** (5–8 líneas) ¿Cuántos puntos de exactitud ganaste al pasar de 16 a 64 ejemplos por clase, y cuántos de 64 a 256? ¿Qué forma tiene tu curva y por qué crees que tiene esa forma? Si etiquetar cada ejemplo costara 1 minuto de un analista, ¿valdría la pena pasar de 256 a 1.024 ejemplos por clase para este problema? Si tu `k_train` no es 1.024, estima o extrapola a partir de la forma de tu curva. Usa tus números.

## Tarea 2 — Few-shot con un LLM frente a tu clasificador (Ejecución técnica + Análisis)

Clasifica las **primeras `N_TEST_LLM` = 40 noticias de tu corte** con `Qwen2.5-0.5B-Instruct` en modo zero-shot (solo instrucción) y few-shot (con tus `n_shots` ejemplos por clase en el prompt), y compáralo con la regresión logística sobre **las mismas 40 noticias**.

1. Ejecuta la celda (en CPU puede tardar unos minutos; con GPU, segundos).
2. ✏️ Completa la tabla: agrega la fila "Embeddings + LR (mismos ejemplos del prompt)", entrenando la regresión logística con exactamente los `few_shot_examples` que vio el LLM.
3. Revisa las respuestas crudas del LLM y responde.

In [ ]:
llm_texts, llm_y = test_texts[:N_TEST_LLM], y_test[:N_TEST_LLM]
few_shot_examples = list(zip(*sample_per_class(ag_news["train"], N_SHOTS, seed=PERSONAL_SEED % (2**32) + 1)))
random.Random(PERSONAL_SEED).shuffle(few_shot_examples)
# Muestra de 1.000 valores de cada tensor de pesos del LLM, antes de usarlo (la usarás en la Tarea 4)
weights_sample_before = torch.cat([p.detach().flatten()[:1000].cpu().clone() for p in llm.parameters()])

zs_labels, zs_answers, zs_seconds, zs_tokens = llm_classify(llm_texts)
fs_labels, fs_answers, fs_seconds, fs_tokens = llm_classify(llm_texts, few_shot_examples)
acc_zero_shot = float(np.mean([p == LABELS[y] for p, y in zip(zs_labels, llm_y)]))
acc_few_shot = float(np.mean([p == LABELS[y] for p, y in zip(fs_labels, llm_y)]))
acc_lr_llm_subset = accuracy_score(llm_y, clf_k.predict(X_test[:N_TEST_LLM]))

# Latencia de la regresión logística: embedding + predicción de un texto
start = time.perf_counter()
clf_k.predict(embed(llm_texts))
ms_lr = 1000 * (time.perf_counter() - start) / len(llm_texts)

comparison = [
    {"estrategia": "Zero-shot LLM", "etiquetas usadas": 0, "exactitud": acc_zero_shot,
     "ms por noticia": 1000 * np.mean(zs_seconds), "tokens de prompt": np.mean(zs_tokens),
     "unknown": zs_labels.count("unknown")},
    {"estrategia": f"Few-shot LLM ({N_SHOTS} por clase)", "etiquetas usadas": len(few_shot_examples),
     "exactitud": acc_few_shot, "ms por noticia": 1000 * np.mean(fs_seconds),
     "tokens de prompt": np.mean(fs_tokens), "unknown": fs_labels.count("unknown")},
    {"estrategia": f"Embeddings + LR (k_train = {K_TRAIN})", "etiquetas usadas": N_CLASSES * K_TRAIN,
     "exactitud": acc_lr_llm_subset, "ms por noticia": ms_lr, "tokens de prompt": 0, "unknown": 0},
]
# ✏️ TU CÓDIGO AQUÍ: agrega la fila "Embeddings + LR (mismos ejemplos del prompt)".
#    Pista: entrena LogisticRegression con embed([t for t, _ in few_shot_examples]) y sus etiquetas,
#    y evalúa con X_test[:N_TEST_LLM] y llm_y. Guarda la exactitud en acc_lr_mismos_ejemplos.
acc_lr_mismos_ejemplos = None

comparison_df = pd.DataFrame(comparison).set_index("estrategia")
display(comparison_df.round({"exactitud": 3, "ms por noticia": 1, "tokens de prompt": 0}))

# Error estándar aproximado de una exactitud medida en n ejemplos: sqrt(p·(1−p)/n)
se_few_shot = float(np.sqrt(acc_few_shot * (1 - acc_few_shot) / N_TEST_LLM))
print(f"Error estándar aproximado de la exactitud few-shot con {N_TEST_LLM} noticias: ±{se_few_shot:.3f}")

pd.DataFrame({"noticia": [shorten(t, 10) for t in llm_texts], "real": [LABELS[y] for y in llm_y],
              "zero-shot": zs_answers, "few-shot": fs_answers}).head(12)

**Tu respuesta:** (6–10 líneas) Con tus números: ¿mejoró el LLM al pasar de zero-shot a few-shot? ¿Cómo se compara con la regresión logística de tu `k_train` y con la entrenada con los mismos ejemplos del prompt? ¿Es la diferencia mayor que el error estándar? ¿Cuántas veces más lenta es cada noticia con el LLM, y cuántos tokens de prompt agregaron tus ejemplos? ¿Qué patrón ves en las respuestas crudas (etiquetas que el modelo sobreusa, respuestas `unknown`)?

## Tarea 3 — Explica y decide (Análisis e interpretación)

Tu caso de negocio está en `CONFIG["business_case"]`. Ejecuta la celda para ver el caso y una estimación de volumen con **tus** mediciones de la Tarea 2 (tokens de prompt y milisegundos por texto). El precio es un valor ilustrativo: si quieres usar uno real, busca el precio actual de un proveedor y cita la fecha.

In [ ]:
case = CONFIG["business_case"]
for key, value in case.items():
    print(f"{key:18s}: {value}")

PRICE_PER_1K_TOKENS = 0.001      # USD, valor ilustrativo (reemplázalo por un precio actual si lo buscas)
SPANISH_FACTOR = 1.5             # ✏️ ajusta: ¿cuántos tokens más ocupa el español? (lab, Parte 1)
volume = case["volumen_mensual"]
tokens_month = volume * np.mean(fs_tokens) * SPANISH_FACTOR
llm_hours = volume * np.mean(fs_seconds) / 3600
lr_hours = volume * ms_lr / 1000 / 3600
print(f"\nSi usaras tu few-shot con un LLM vía API: ≈ {tokens_month / 1e6:.1f} M tokens de entrada al mes "
      f"≈ USD {tokens_month / 1000 * PRICE_PER_1K_TOKENS:,.2f} con el precio ilustrativo")
print(f"Tiempo de cómputo al mes en tu máquina ({DEVICE}): LLM ≈ {llm_hours:.1f} h · embeddings + LR ≈ {lr_hours:.2f} h")

**Tu respuesta:** Recomienda una estrategia de transferencia para tu caso. Usa esta estructura (8–12 líneas en total):
1. **Estrategia recomendada** (zero-shot, few-shot, extracción de características, fine-tuning parcial o completo, adaptación de dominio) y **por qué**, citando al menos dos números tuyos (curva de aprendizaje, exactitud few-shot, latencia, tokens).
2. **Modelo de partida:** tamaño, idioma (¿inglés, multilingüe o en español, como BETO?) y licencia que buscarías en la *model card*.
3. **Un riesgo** de tu recomendación y cómo lo medirías antes de salir a producción.
4. **Qué te haría cambiar de opinión:** el número que, si lo observaras, te haría subir o bajar un escalón en el espectro de transferencia.

## Tarea 4 — Crítica a la IA (Análisis e interpretación)

Un asistente de IA escribió esta explicación para un informe interno. **Tiene exactamente 3 errores conceptuales.** Encuéntralos, explica por qué están mal y corrígelos. La celda de código te ayuda a comprobar dos de ellos con evidencia.

> *"En un Transformer, cada token se convierte primero en un vector (embedding) al que se le suma información de posición, porque la atención por sí sola no distingue el orden de las palabras. Luego, la autoatención calcula para cada token unos pesos sobre todos los demás con la fórmula softmax(QKᵀ · √d_k)·V: se multiplica por √d_k para amplificar las diferencias entre puntuaciones y que el modelo se concentre en las palabras importantes. Cada fila de pesos suma 1. BERT es un modelo decodificador que se preentrena prediciendo la siguiente palabra de izquierda a derecha; por eso su vector [CLS] resume bien la frase y es ideal para generar textos largos. GPT, en cambio, usa una máscara causal: cada token solo atiende a los tokens anteriores. En transferencia de aprendizaje, la extracción de características congela el codificador y entrena solo un clasificador pequeño encima, por lo que funciona incluso en CPU. Con few-shot, los ejemplos del prompt ajustan los pesos del modelo con un pequeño paso de gradiente, así que después de verlos el modelo recuerda la tarea en las conversaciones siguientes. Finalmente, como los tokenizadores entrenados principalmente con inglés parten el español en más tokens, el mismo contenido en español suele costar más en una API que cobra por token."*

In [ ]:
# Evidencia 1: ¿qué pasa con los pesos de atención si se DIVIDE o se MULTIPLICA por √d_k?
def max_attention_weight(d_k, scale, n_tokens=10, repeats=200, seed=0):
    """Promedio del peso máximo por fila de softmax(QKᵀ ⋅ escala) con Q y K aleatorios (varianza 1)."""
    gen = torch.Generator().manual_seed(seed)
    q = torch.randn(repeats, n_tokens, d_k, generator=gen)
    k = torch.randn(repeats, n_tokens, d_k, generator=gen)
    scores = q @ k.transpose(-1, -2)
    factor = {"sin escalar": 1.0, "dividir": d_k ** -0.5, "multiplicar": d_k ** 0.5}[scale]
    return torch.softmax(scores * factor, dim=-1).max(dim=-1).values.mean().item()

scaling_rows = [{"d_k": d, **{s: round(max_attention_weight(d, s), 3) for s in ["dividir", "sin escalar", "multiplicar"]}}
                for d in [16, 64, 256]]
display(pd.DataFrame(scaling_rows).set_index("d_k"))
print("Con n_tokens = 10, un peso máximo cercano a 1.0 significa que la softmax está saturada (casi one-hot).")

# Evidencia 2: ¿cambiaron los pesos del LLM después de las llamadas few-shot de la Tarea 2?
weights_sample_after = torch.cat([p.detach().flatten()[:1000].cpu() for p in llm.parameters()])
changed = int((weights_sample_before != weights_sample_after).sum())
print(f"\nValores de pesos revisados: {weights_sample_after.numel():,} · valores que cambiaron tras "
      f"{2 * N_TEST_LLM} llamadas (zero-shot + few-shot): {changed}")
# ✏️ TU CÓDIGO AQUÍ (opcional): otra evidencia, p. ej., vuelve a correr zero-shot y compara con zs_labels.

**Tu respuesta:** completa la tabla (una fila por error) y explica en 2–3 líneas qué muestran las dos evidencias de la celda anterior.

| # | Frase con el error (cítala) | Por qué está mal | Versión corregida |
|---|---|---|---|
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

In [ ]:
RESULTS = {
    "k_train": int(K_TRAIN),
    "n_shots": int(N_SHOTS),
    "n_test": int(N_TEST),
    "n_test_llm": int(N_TEST_LLM),
    "curva_aprendizaje": {str(k): round(float(a), 4) for k, a in curve.items()},
    "acc_lr_k_train": round(float(acc_lr_k), 4),
    "acc_lr_k_train_40": round(float(acc_lr_llm_subset), 4),
    "acc_zero_shot_40": round(acc_zero_shot, 4),
    "acc_few_shot_40": round(acc_few_shot, 4),
    "acc_lr_mismos_ejemplos_40": None if acc_lr_mismos_ejemplos is None else round(float(acc_lr_mismos_ejemplos), 4),
    "unknown_few_shot": int(fs_labels.count("unknown")),
    "tokens_prompt_few_shot": round(float(np.mean(fs_tokens)), 1),
    "ms_por_noticia_lr": round(float(ms_lr), 4),
    "ms_por_noticia_few_shot": round(float(1000 * np.mean(fs_seconds)), 4),
    "dispositivo": DEVICE,
}
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))